In [1]:
from pyspark.sql import SparkSession
import pandas as pd
from pyspark.sql.types import IntegerType, StringType, DoubleType, StructField, StructType

In [2]:
import sys
import os
import logging
sys.executable

'/home/bakerville/Repositories/spark-on-docker/venv/bin/python'

In [ ]:
# spark.stop()

spark = SparkSession.builder.master("spark://spark-master:7077")\
                .config("spark.driver.extraClassPath", "./jars/sqljdbc42.jar")\
                .config('spark.executor.extraClassPath', "./sqljdbc42.jar")\
                .config("spark.jars", "./jars/sqljdbc42.jar")\
                .appName("SparkETL.com")\
                .getOrCreate()


26/03/25 00:09:10 WARN Utils: Your hostname, bakerville-linux-system resolves to a loopback address: 127.0.1.1; using 192.168.1.208 instead (on interface wlp0s20f3)
26/03/25 00:09:10 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
26/03/25 00:09:10 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/25 00:09:10 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/03/25 00:09:13 WARN StandaloneAppClient$ClientEndpoint: Failed to connect to master 172.17.0.2:7077
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:56)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:310)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(

26/03/25 00:10:11 WARN StandaloneAppClient$ClientEndpoint: Drop UnregisterApplication(null) because has not yet connected to master


In [10]:
spark.sparkContext.getConf().getAll()

[('spark.driver.extraJavaOptions',
  '-Djava.net.preferIPv6Addresses=false -XX:+IgnoreUnrecognizedVMOptions --add-opens=java.base/java.lang=ALL-UNNAMED --add-opens=java.base/java.lang.invoke=ALL-UNNAMED --add-opens=java.base/java.lang.reflect=ALL-UNNAMED --add-opens=java.base/java.io=ALL-UNNAMED --add-opens=java.base/java.net=ALL-UNNAMED --add-opens=java.base/java.nio=ALL-UNNAMED --add-opens=java.base/java.util=ALL-UNNAMED --add-opens=java.base/java.util.concurrent=ALL-UNNAMED --add-opens=java.base/java.util.concurrent.atomic=ALL-UNNAMED --add-opens=java.base/jdk.internal.ref=ALL-UNNAMED --add-opens=java.base/sun.nio.ch=ALL-UNNAMED --add-opens=java.base/sun.nio.cs=ALL-UNNAMED --add-opens=java.base/sun.security.action=ALL-UNNAMED --add-opens=java.base/sun.util.calendar=ALL-UNNAMED --add-opens=java.security.jgss/sun.security.krb5=ALL-UNNAMED -Djdk.reflect.useDirectMethodHandle=false'),
 ('spark.jars', './jars/sqljdbc42.jar'),
 ('spark.master', 'local[4]'),
 ('spark.repl.local.jars',
  'f

In [12]:
source_schema = StructType([
        StructField("ID", IntegerType(), False),
        StructField("USERNAME", StringType(), False),
        StructField("NUMBER_FOLLOWS", StringType(), False),
        StructField("NUMBER_TRACKS", StringType(), False),
        StructField("LINK", StringType(), False)])

In [13]:
if os.path.exists("./data/"):
    source_df = spark.read.options(delimiter=',').csv("./data/Soundcloud_User.csv", schema=source_schema,header=True)
    source_df.show()

+---+--------------------+----------------+-------------+--------------------+
| ID|            USERNAME|  NUMBER_FOLLOWS|NUMBER_TRACKS|                LINK|
+---+--------------------+----------------+-------------+--------------------+
|  0|                   a|14,992 followers|            0|https://soundclou...|
|  1|            Hoodadk4|18,936 followers|            0|https://soundclou...|
|  2|         Dani Ripoll| 8,386 followers|            0|https://soundclou...|
|  3|      4Traxx500Music|33,698 followers|            0|https://soundclou...|
|  4|        DJ KHALID- A|29,959 followers|            0|https://soundclou...|
|  5|       Dj PixelStorm| 6,462 followers|            0|https://soundclou...|
|  6|               Erfan|73,327 followers|            0|https://soundclou...|
|  7|           c4pri (4)| 3,896 followers|            0|https://soundclou...|
|  8|مهرجانات الجديدة ...| 2,364 followers|            0|https://soundclou...|
|  9|         §noah kwed§|12,735 followers|         

26/03/24 23:58:58 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , users_name, num_followers, num_tracks, users_link
 Schema: ID, USERNAME, NUMBER_FOLLOWS, NUMBER_TRACKS, LINK
Expected: ID but found: 
CSV file: file:///home/bakerville/Repositories/spark-on-docker/pyspark_etl/data/Soundcloud_User.csv


In [12]:
from dotenv import load_dotenv, dotenv_values

USERNAME = dotenv_values(".env").get("USERNAME")
HOST = dotenv_values(".env").get("HOST")
CONNECTION_STRING = dotenv_values(".env").get("CONNECTION_STRING")
PASSWORD = dotenv_values(".env").get("PASSWORD")
TABLE_NAME = 'CustomerSource'
DATABASE = dotenv_values(".env").get("DATABASE")

logging.info(f"USERNAME: {USERNAME} HOST: {HOST}  PASSWORD: {PASSWORD}")

url = f"jdbc:postgresql://{HOST}:5432/{DATABASE}"
properties = {
    "user": dotenv_values('.env').get('USERNAME'),
    "password": dotenv_values('.env').get('PASSWORD'),
    "driver": "org.postgresql.Driver"
}

In [ ]:
import psycopg2

# Connect to PostgreSQL
try:
    conn = psycopg2.connect(host=HOST, user=USERNAME, password=PASSWORD, database="postgres")
    cursor = conn.cursor()
    cursor.execute("SELECT 1 FROM pg_database WHERE datname = 'spark_db'")
    if not cursor.fetchone():
        cursor.execute("CREATE DATABASE spark_db")
        print("Database 'spark_db' created")
    else:
        print("Database 'spark_db' already exists")
    cursor.close()
    conn.close()
except Exception as e:
    print(f"Error: {e}")

In [8]:
df = spark.read \
  .format("jdbc") \
  .option("driver","org.postgresql.Driver") \
  .option("url", url) \
  .option("dbtable", "soundclouduser") \
  .option("user", USERNAME) \
  .option("password", PASSWORD) \
  .load()

NameError: name 'spark' is not defined

In [9]:
df.show()

+---+--------------------+-------------+------------+--------------------+
| Id|            Username|NumberFollows|NumberTracks|                Link|
+---+--------------------+-------------+------------+--------------------+
|  0|                   a|        14973|           0|https://soundclou...|
|  1|            Hoodadk4|        15935|           0|https://soundclou...|
|  2|           Sexyy Red|        45021|           0|https://soundclou...|
|  3|               4batz|        34867|           0|https://soundclou...|
|  4|         Sara Landry|        86088|           0|https://soundclou...|
|  5|                AKON|       238447|           0|https://soundclou...|
|  6|A BOOGIE WIT DA H...|      1587348|           0|https://soundclou...|
|  7|           21 Savage|      1671201|           0|https://soundclou...|
|  8|MC IG - 4M (CONTA...|         8449|           0|https://soundclou...|
|  9|      Nico Morrisson|         9158|           0|https://soundclou...|
| 10| Federico Maniscalco

In [10]:
list(source_schema)

[StructField('ID', IntegerType(), False),
 StructField('USERNAME', StringType(), False),
 StructField('NUMBER_FOLLOWS', StringType(), False),
 StructField('NUMBER_TRACKS', StringType(), False),
 StructField('LINK', StringType(), False)]

In [7]:
source_df.write \
  .format("jdbc") \
  .option("mode","overwrite")\
  .option("driver","org.postgresql.Driver") \
  .option("url", url) \
  .option("dbtable", TABLE_NAME) \
  .option("user", USERNAME) \
  .option("password", PASSWORD)\
  .save(mode="overwrite")

NameError: name 'source_df' is not defined

In [12]:
source_df.show()

+---+--------------------+-------------------+-------------+--------------------+
| ID|            USERNAME|     NUMBER_FOLLOWS|NUMBER_TRACKS|                LINK|
+---+--------------------+-------------------+-------------+--------------------+
|  0|                   a|   14,973 followers|            0|https://soundclou...|
|  1|            Hoodadk4|   15,935 followers|            0|https://soundclou...|
|  2|           Sexyy Red|   45,021 followers|            0|https://soundclou...|
|  3|               4batz|   34,867 followers|            0|https://soundclou...|
|  4|         Sara Landry|   86,088 followers|            0|https://soundclou...|
|  5|                AKON|  238,447 followers|            0|https://soundclou...|
|  6|A BOOGIE WIT DA H...|1,587,348 followers|            0|https://soundclou...|
|  7|           21 Savage|1,671,201 followers|            0|https://soundclou...|
|  8|MC IG - 4M (CONTA...|    8,449 followers|            0|https://soundclou...|
|  9|      Nico 

In [13]:

from pyspark.sql.functions import split, regexp_replace
from pyspark.sql.functions import col

def extract_number_follow(value):
    return split(value," ").getItem(0)
df=source_df.withColumn("NEW_NUMBER_FOLLOWS",extract_number_follow(source_df.NUMBER_FOLLOWS))
df.show()


+---+--------------------+-------------------+-------------+--------------------+------------------+
| ID|            USERNAME|     NUMBER_FOLLOWS|NUMBER_TRACKS|                LINK|NEW_NUMBER_FOLLOWS|
+---+--------------------+-------------------+-------------+--------------------+------------------+
|  0|                   a|   14,973 followers|            0|https://soundclou...|            14,973|
|  1|            Hoodadk4|   15,935 followers|            0|https://soundclou...|            15,935|
|  2|           Sexyy Red|   45,021 followers|            0|https://soundclou...|            45,021|
|  3|               4batz|   34,867 followers|            0|https://soundclou...|            34,867|
|  4|         Sara Landry|   86,088 followers|            0|https://soundclou...|            86,088|
|  5|                AKON|  238,447 followers|            0|https://soundclou...|           238,447|
|  6|A BOOGIE WIT DA H...|1,587,348 followers|            0|https://soundclou...|         1

In [14]:
df.createOrReplaceTempView("soundcloud")

In [15]:
df_2 = spark.sql("""
    Select *, cast(replace(new_number_follows,',','') as INT) as DB_FOLLOWS
    from soundcloud

""")

df_2.createOrReplaceTempView("transformed_data")

In [16]:
destination = spark.sql("""SELECT username, db_follows, number_tracks, link from transformed_data """)

In [17]:
destination.show()

+--------------------+----------+-------------+--------------------+
|            username|db_follows|number_tracks|                link|
+--------------------+----------+-------------+--------------------+
|                   a|     14973|            0|https://soundclou...|
|            Hoodadk4|     15935|            0|https://soundclou...|
|           Sexyy Red|     45021|            0|https://soundclou...|
|               4batz|     34867|            0|https://soundclou...|
|         Sara Landry|     86088|            0|https://soundclou...|
|                AKON|    238447|            0|https://soundclou...|
|A BOOGIE WIT DA H...|   1587348|            0|https://soundclou...|
|           21 Savage|   1671201|            0|https://soundclou...|
|MC IG - 4M (CONTA...|      8449|            0|https://soundclou...|
|      Nico Morrisson|      9158|            0|https://soundclou...|
| Federico Maniscalco|      8962|            0|https://soundclou...|
|            PK Sonik|      8067| 

In [18]:
destination.schema

StructType([StructField('username', StringType(), True), StructField('db_follows', IntegerType(), True), StructField('number_tracks', StringType(), True), StructField('link', StringType(), True)])

In [19]:
from packages.ExtractLoad import ExtractLoadFiletoDB
from packages.Transform import Transform
from dotenv import load_dotenv, dotenv_values
from pyspark.sql.types import IntegerType, StringType, DoubleType, StructField, StructType

source_schema = StructType([
        StructField("ID", IntegerType(), False),
        StructField("USERNAME", StringType(), False),
        StructField("NUMBER_FOLLOWS",StringType(), False),
        StructField("NUMBER_TRACKS", IntegerType(), False),
        StructField("LINK", StringType(), False)])
        
FILEPATH = "Soundcloud_User.csv"

USERNAME = dotenv_values(".env").get("USERNAME")
HOST = dotenv_values(".env").get("HOST")
CONNECTION_STRING = dotenv_values(".env").get("CONNECTION_STRING")
PASSWORD = dotenv_values(".env").get("PASSWORD")

URL = dotenv_values(".env").get("URL")

# Extract data from local and create a spark session
IngestProcess = ExtractLoadFiletoDB()

IngestProcess.createSparkSession(masterName="local[1]", appName="ETLSpark.com", driverPos="./jars/sqljdbc42.jar")

IngestProcess.extractSourceFile(filePath=FILEPATH, delimiter=",", schema=source_schema)

source = IngestProcess.sourceDataFrame

spark = IngestProcess.spark

# Transform daa
TransformProcess = Transform(data= source, spark=spark)

temp = TransformProcess.string_to_int(source, "NUMBER_FOLLOWS", "DB_NUMBER_FOLLOWS")

temp_1 = TransformProcess.string_to_int(temp, "NUMBER_TRACKS", "DB_NUMBER_TRACKS")

temp_2 = temp_1.withColumnRenamed("db_number_follows","NumberFollows").withColumnRenamed("db_number_tracks","NumberTracks")

destination = temp_2.select("Id","Username", "NumberFollows", "NumberTracks", "Link")

#Load data to table in database
IngestProcess.createDBConnection(url = URL, username=USERNAME, password=PASSWORD)

IngestProcess.writeDBTable(tablename="[dbo].[SoundCloudUser]", dataframe=destination)



In [20]:
destination.withColumnRenamed("db_number_follows","NumberFollows").withColumnRenamed("db_number_tracks","NumberTracks").show()

+---+--------------------+-------------+------------+--------------------+
| Id|            Username|NumberFollows|NumberTracks|                Link|
+---+--------------------+-------------+------------+--------------------+
|  0|                   a|        14973|           0|https://soundclou...|
|  1|            Hoodadk4|        15935|           0|https://soundclou...|
|  2|           Sexyy Red|        45021|           0|https://soundclou...|
|  3|               4batz|        34867|           0|https://soundclou...|
|  4|         Sara Landry|        86088|           0|https://soundclou...|
|  5|                AKON|       238447|           0|https://soundclou...|
|  6|A BOOGIE WIT DA H...|      1587348|           0|https://soundclou...|
|  7|           21 Savage|      1671201|           0|https://soundclou...|
|  8|MC IG - 4M (CONTA...|         8449|           0|https://soundclou...|
|  9|      Nico Morrisson|         9158|           0|https://soundclou...|
| 10| Federico Maniscalco